# 06 — Siamese ResNet: Tumor + Healthy

## Obiettivo

Migliorare la capacità rappresentativa della Siamese Network mantenendo
inalterato il paradigma pairwise richiesto dal tutor.

Gli esperimenti precedenti hanno mostrato che:

- Euclidean Contrastive Loss è stabile;
- Cosine Similarity produce uno spazio metrico meno discriminativo;
- Batch-Hard Triplet Loss può portare a representation collapse;
- SupCon migliora leggermente la classificazione multiclass;
- metodi ausiliari supervisionati migliorano ulteriormente, ma si
  allontanano dalla formulazione Siamese pairwise originale.

In questo esperimento torniamo quindi alla pipeline Siamese pura:

FCGR A ──→ shared CNN encoder ──→ embedding A
                                     │
                                     ├── Euclidean Distance
                                     │
FCGR B ──→ shared CNN encoder ──→ embedding B
                                     ↓
                              Contrastive Loss

La modifica principale riguarda l'encoder.

La CNN V3 sequenziale viene sostituita con una CNN residuale con
Residual Blocks e GroupNorm.

## Configurazione

- Task: 12 classi Tumor + Healthy
- FCGR: k=6
- input: 64 × 64
- downsampling: 2111 sample per classe
- coppie positive/negative ≈ 50/50
- shared Residual CNN
- embedding: 128D L2-normalizzato
- Euclidean Contrastive Loss
- margin = 1.25
- split basato su split_cluster
- test set non utilizzato

## Obiettivo sperimentale

Confrontare direttamente:

CNN V3 + Euclidean Contrastive

vs

Residual CNN + Euclidean Contrastive

per verificare se il limite osservato finora dipende dalla capacità
dell'encoder di estrarre pattern discriminativi dalle FCGR.

In [1]:
# ============================================================
# CELL 2 — IMPORT
# ============================================================

from pathlib import Path

import json
import random
import time
import copy
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

PyTorch: 2.12.0+cu126
CUDA: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ============================================================
# CELL 3 — PATH + CONFIG
# ============================================================

CURRENT_DIR = Path.cwd().resolve()


if CURRENT_DIR.name == "notebooks":

    PROJECT_ROOT = CURRENT_DIR.parent

else:

    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_resnet_tumor_healthy"
)


ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)


with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    pair_config = json.load(f)


K = int(
    pair_config["k"]
)


RANDOM_STATE = int(
    pair_config["random_state"]
)


TRAIN_PAIRS_PER_EPOCH = int(
    pair_config["train_pairs_per_epoch"]
)


VAL_PAIRS = int(
    pair_config["val_pairs"]
)


FCGR_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


# ============================================================
# MODEL
# ============================================================

N_CLASSES = 12

EMBEDDING_DIM = 128

EUCLIDEAN_MARGIN = 1.25

POSITIVE_PROBABILITY = 0.5


BATCH_SIZE = 128


print(
    "Project:",
    PROJECT_ROOT
)

print(
    "k:",
    K
)

print(
    "Train pairs/epoch:",
    TRAIN_PAIRS_PER_EPOCH
)

print(
    "Validation pairs:",
    VAL_PAIRS
)

print(
    "Pair batch size:",
    BATCH_SIZE
)

Project: D:\Daria\Desktop\eccdna_fcgr_siamese
k: 6
Train pairs/epoch: 50000
Validation pairs: 10000
Pair batch size: 128


In [3]:
# ============================================================
# CELL 4 — REPRODUCIBILITY + DEVICE
# ============================================================

def set_seed(
    seed
):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    RANDOM_STATE
)


DEVICE = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


AMP_ENABLED = (
    DEVICE.type == "cuda"
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


print(
    "Device:",
    DEVICE
)

print(
    "AMP:",
    AMP_ENABLED
)

Device: cuda
AMP: True


In [4]:
# ============================================================
# CELL 5 — TUMOR + HEALTHY + CLEAN DOWNSAMPLING
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


full_train_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# SMALLEST CLASS
# ============================================================

train_counts = (
    full_train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
)


min_train_class_size = int(
    train_counts.min()
)


# ============================================================
# SAME STATIC DOWNSAMPLING USED IN 04h
# ============================================================

balanced_parts = []


for class_id in sorted(
    full_train_metadata[
        "class_id"
    ].unique()
):

    class_data = (
        full_train_metadata[
            full_train_metadata[
                "class_id"
            ]
            ==
            class_id
        ]
    )


    class_sample = class_data.sample(

        n=min_train_class_size,

        replace=False,

        random_state=(
            RANDOM_STATE
            +
            int(class_id)
        )
    )


    balanced_parts.append(
        class_sample
    )


train_metadata = (
    pd.concat(
        balanced_parts,
        ignore_index=True
    )
    .reset_index(drop=True)
)


print("=" * 72)
print("TRAIN DOWNSAMPLING")
print("=" * 72)

print(
    "Full train:",
    len(full_train_metadata)
)

print(
    "Classe minima:",
    min_train_class_size
)

print(
    "Balanced train:",
    len(train_metadata)
)

print(
    "N classi:",
    train_metadata[
        "class_id"
    ].nunique()
)

print()


display(

    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_train")
    .to_frame()
)

TRAIN DOWNSAMPLING
Full train: 96167
Classe minima: 2111
Balanced train: 25332
N classi: 12



,n_train
class_id,
0,2111
1,2111
2,2111
3,2111
4,2111
5,2111
6,2111
7,2111
8,2111


In [5]:
# ============================================================
# CELL 6 — VALIDATION POOL
# ============================================================

class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


old_to_new = dict(

    zip(

        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


included_original_ids = set(
    old_to_new.keys()
)


val_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_original["class_id"] = (
    val_original["class_id"]
    .astype(int)
)


val_metadata = (
    val_original[
        val_original[
            "class_id"
        ].isin(
            included_original_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = (
    val_metadata[
        "class_id"
    ]
)


val_metadata[
    "class_id"
] = (

    val_metadata[
        "original_class_id"
    ]
    .map(
        old_to_new
    )
    .astype(int)
)


print(
    "Validation samples:",
    len(val_metadata)
)

print(
    "Validation classes:",
    val_metadata[
        "class_id"
    ].nunique()
)


display(

    val_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_val")
    .to_frame()
)

Validation samples: 9753
Validation classes: 12


,n_val
class_id,
0,1000
1,1000
2,1000
3,1000
4,1000
5,1000
6,1000
7,1000
8,585


In [6]:
# ============================================================
# CELL 7 — FCGR MEMMAP
# ============================================================

fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(

    zip(

        fcgr_index["id"],

        fcgr_index["fcgr_row"]
    )
)


missing_train = (

    ~train_metadata[
        "id"
    ].isin(
        id_to_fcgr_row
    )

).sum()


missing_val = (

    ~val_metadata[
        "id"
    ].isin(
        id_to_fcgr_row
    )

).sum()


print(
    "FCGR shape:",
    fcgr_memmap.shape
)

print(
    "FCGR dtype:",
    fcgr_memmap.dtype
)

print(
    "Missing train:",
    missing_train
)

print(
    "Missing val:",
    missing_val
)


assert missing_train == 0

assert missing_val == 0

FCGR shape: (150272, 64, 64)
FCGR dtype: float32
Missing train: 0
Missing val: 0


In [7]:
# ============================================================
# CELL 8 — SIAMESE PAIR DATASET
# ============================================================

class SiamesePairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        n_pairs,
        positive_fraction=0.5,
        seed=42,
        dynamic=False
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )


        self.fcgr_memmap = (
            fcgr_memmap
        )


        self.n_pairs = int(
            n_pairs
        )


        self.positive_fraction = float(
            positive_fraction
        )


        self.seed = int(
            seed
        )


        self.dynamic = bool(
            dynamic
        )


        # ----------------------------------------------------
        # FCGR row
        # ----------------------------------------------------

        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata[
                "class_id"
            ]
            .to_numpy(dtype=np.int64)
        )


        self.classes = np.array(

            sorted(
                np.unique(
                    self.labels
                )
            ),

            dtype=np.int64
        )


        self.class_to_indices = {

            int(class_id):

                np.where(
                    self.labels
                    ==
                    class_id
                )[0]

            for class_id
            in self.classes
        }


        self.epoch = 0


        self._generate_pairs(
            self.seed
        )


    def _generate_pairs(
        self,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )


        n_positive = int(

            round(
                self.n_pairs
                *
                self.positive_fraction
            )
        )


        targets = np.zeros(
            self.n_pairs,
            dtype=np.float32
        )


        targets[
            :n_positive
        ] = 1.0


        rng.shuffle(
            targets
        )


        anchors = rng.integers(

            low=0,

            high=len(
                self.labels
            ),

            size=self.n_pairs
        )


        partners = np.empty(
            self.n_pairs,
            dtype=np.int64
        )


        for i in range(
            self.n_pairs
        ):

            anchor_idx = int(
                anchors[i]
            )


            anchor_class = int(
                self.labels[
                    anchor_idx
                ]
            )


            # =================================================
            # POSITIVE
            # =================================================

            if targets[i] == 1.0:

                candidates = (

                    self.class_to_indices[
                        anchor_class
                    ]
                )


                partner_idx = (
                    anchor_idx
                )


                while (
                    partner_idx
                    ==
                    anchor_idx
                ):

                    partner_idx = int(

                        rng.choice(
                            candidates
                        )
                    )


            # =================================================
            # NEGATIVE
            # =================================================

            else:

                negative_classes = (

                    self.classes[
                        self.classes
                        !=
                        anchor_class
                    ]
                )


                negative_class = int(

                    rng.choice(
                        negative_classes
                    )
                )


                partner_idx = int(

                    rng.choice(

                        self.class_to_indices[
                            negative_class
                        ]
                    )
                )


            partners[i] = (
                partner_idx
            )


        self.row1 = (
            self.rows[
                anchors
            ]
        )


        self.row2 = (
            self.rows[
                partners
            ]
        )


        self.targets = (
            targets
        )


    def set_epoch(
        self,
        epoch
    ):

        self.epoch = int(
            epoch
        )


        if self.dynamic:

            self._generate_pairs(

                self.seed
                +
                self.epoch
                *
                100_003
            )


    def __len__(
        self
    ):

        return (
            self.n_pairs
        )


    def __getitem__(
        self,
        index
    ):

        row1 = int(
            self.row1[index]
        )

        row2 = int(
            self.row2[index]
        )


        x1 = np.array(

            self.fcgr_memmap[
                row1
            ],

            dtype=np.float32,

            copy=True
        )


        x2 = np.array(

            self.fcgr_memmap[
                row2
            ],

            dtype=np.float32,

            copy=True
        )


        return {

            "x1":
                torch.from_numpy(
                    x1
                ).unsqueeze(0),

            "x2":
                torch.from_numpy(
                    x2
                ).unsqueeze(0),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.float32
                )
        }

In [8]:
# ============================================================
# CELL 9 — PAIR LOADERS
# ============================================================

train_pair_dataset = SiamesePairDataset(

    metadata=
        train_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row,

    n_pairs=
        TRAIN_PAIRS_PER_EPOCH,

    positive_fraction=
        POSITIVE_PROBABILITY,

    seed=
        RANDOM_STATE,

    dynamic=True
)


val_pair_dataset = SiamesePairDataset(

    metadata=
        val_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row,

    n_pairs=
        VAL_PAIRS,

    positive_fraction=
        0.5,

    seed=
        RANDOM_STATE
        +
        50_000,

    dynamic=False
)


train_pair_loader = DataLoader(

    train_pair_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


val_pair_loader = DataLoader(

    val_pair_dataset,

    batch_size=
        BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Train pairs:",
    len(train_pair_dataset)
)

print(
    "Validation pairs:",
    len(val_pair_dataset)
)

print(
    "Train positive fraction:",
    train_pair_dataset.targets.mean()
)

print(
    "Val positive fraction:",
    val_pair_dataset.targets.mean()
)


batch_check = next(
    iter(train_pair_loader)
)


print()
print(
    "x1:",
    batch_check["x1"].shape
)

print(
    "x2:",
    batch_check["x2"].shape
)

print(
    "targets:",
    batch_check["target"].shape
)

Train pairs: 50000
Validation pairs: 10000
Train positive fraction: 0.5
Val positive fraction: 0.5

x1: torch.Size([128, 1, 64, 64])
x2: torch.Size([128, 1, 64, 64])
targets: torch.Size([128])


In [9]:
# ============================================================
# CELL 10 — RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):

        super().__init__()


        # ====================================================
        # MAIN BRANCH
        # ====================================================

        self.conv1 = nn.Conv2d(

            in_channels,
            out_channels,

            kernel_size=3,

            stride=stride,

            padding=1,

            bias=False
        )


        self.norm1 = nn.GroupNorm(

            num_groups=8,

            num_channels=
                out_channels
        )


        self.conv2 = nn.Conv2d(

            out_channels,
            out_channels,

            kernel_size=3,

            stride=1,

            padding=1,

            bias=False
        )


        self.norm2 = nn.GroupNorm(

            num_groups=8,

            num_channels=
                out_channels
        )


        # ====================================================
        # SHORTCUT
        # ====================================================

        if (
            stride != 1
            or
            in_channels != out_channels
        ):

            self.shortcut = nn.Sequential(

                nn.Conv2d(

                    in_channels,
                    out_channels,

                    kernel_size=1,

                    stride=stride,

                    bias=False
                ),

                nn.GroupNorm(

                    num_groups=8,

                    num_channels=
                        out_channels
                )
            )

        else:

            self.shortcut = (
                nn.Identity()
            )


    def forward(
        self,
        x
    ):

        identity = self.shortcut(
            x
        )


        out = self.conv1(
            x
        )

        out = self.norm1(
            out
        )

        out = F.relu(
            out,
            inplace=True
        )


        out = self.conv2(
            out
        )

        out = self.norm2(
            out
        )


        out = (
            out
            +
            identity
        )


        out = F.relu(
            out,
            inplace=True
        )


        return out

In [20]:
# ============================================================
# CELL 11 — COMPACT RESIDUAL FCGR ENCODER
# ============================================================

class FCGRResidualEncoder(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        # ====================================================
        # STEM
        # input: 64 × 64
        # ====================================================

        self.stem = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(
                8,
                32
            ),

            nn.ReLU(
                inplace=True
            )
        )


        # ====================================================
        # STAGE 1
        # 64 × 64
        # ====================================================

        self.stage1 = ResidualBlock(
            32,
            32,
            stride=1
        )


        # ====================================================
        # STAGE 2
        # 64 × 64 → 32 × 32
        # ====================================================

        self.stage2 = ResidualBlock(
            32,
            64,
            stride=2
        )


        # ====================================================
        # STAGE 3
        # 32 × 32 → 16 × 16
        # ====================================================

        self.stage3 = ResidualBlock(
            64,
            128,
            stride=2
        )


        # ====================================================
        # STAGE 4
        # 16 × 16 → 8 × 8
        #
        # Restiamo a 128 canali invece di arrivare a 256.
        # ====================================================

        self.stage4 = ResidualBlock(
            128,
            128,
            stride=2
        )


        # ====================================================
        # GLOBAL POOLING
        # ====================================================

        self.pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )


        # ====================================================
        # EMBEDDING
        # ====================================================

        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128,
                256
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


    def forward(
        self,
        x
    ):

        x = self.stem(x)

        x = self.stage1(x)

        x = self.stage2(x)

        x = self.stage3(x)

        x = self.stage4(x)

        x = self.pool(x)

        z = self.embedding_head(x)


        z = F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )


        return z

In [21]:
# ============================================================
# CELL 12 — SIAMESE RESNET
# ============================================================

class SiameseResidualNetwork(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.encoder = (
            FCGRResidualEncoder(
                embedding_dim=
                    embedding_dim
            )
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = (
            x1.shape[0]
        )


        # Due input, stesso encoder condiviso.
        # Concatenazione solo per efficienza GPU.
        x = torch.cat(

            [
                x1,
                x2
            ],

            dim=0
        )


        z = self.encoder(
            x
        )


        z1 = z[
            :batch_size
        ]


        z2 = z[
            batch_size:
        ]


        return (
            z1,
            z2
        )


model = SiameseResidualNetwork(

    embedding_dim=
        EMBEDDING_DIM

).to(
    DEVICE
)


n_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(
    model
)

print()

print(
    "Trainable parameters:",
    f"{n_params:,}"
)

SiameseResidualNetwork(
  (encoder): FCGRResidualEncoder(
    (stem): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): GroupNorm(8, 32, eps=1e-05, affine=True, bias=True)
      (2): ReLU(inplace=True)
    )
    (stage1): ResidualBlock(
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (norm1): GroupNorm(8, 32, eps=1e-05, affine=True, bias=True)
      (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (norm2): GroupNorm(8, 32, eps=1e-05, affine=True, bias=True)
      (shortcut): Identity()
    )
    (stage2): ResidualBlock(
      (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (norm1): GroupNorm(8, 64, eps=1e-05, affine=True, bias=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (norm2): GroupNorm(8, 64, eps=1e-05, affine=True, bias=True

In [12]:
# ============================================================
# CELL 13 — EUCLIDEAN CONTRASTIVE LOSS
# ============================================================

class EuclideanContrastiveLoss(nn.Module):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        z1,
        z2,
        target
    ):

        z1 = z1.float()

        z2 = z2.float()

        target = target.float()


        distances = torch.linalg.vector_norm(

            z1 - z2,

            ord=2,

            dim=1
        )


        positive_loss = (

            target

            *

            distances.pow(2)
        )


        negative_loss = (

            (1.0 - target)

            *

            F.relu(

                self.margin

                -

                distances

            ).pow(2)
        )


        loss = (

            positive_loss

            +

            negative_loss

        ).mean()


        return (
            loss,
            distances
        )


criterion = EuclideanContrastiveLoss(

    margin=
        EUCLIDEAN_MARGIN
)


print(
    "Margin:",
    criterion.margin
)

Margin: 1.25


In [22]:
# ============================================================
# CELL 14 — RESNET SIAMESE SANITY CHECK
# ============================================================

model.eval()


batch_check = next(
    iter(train_pair_loader)
)


x1 = (
    batch_check["x1"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


x2 = (
    batch_check["x2"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


target = (
    batch_check["target"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


with torch.no_grad():

    with torch.autocast(

        device_type=
            DEVICE.type,

        dtype=(
            torch.float16
            if DEVICE.type == "cuda"
            else torch.bfloat16
        ),

        enabled=
            AMP_ENABLED

    ):

        z1, z2 = model(
            x1,
            x2
        )


    (
        sanity_loss,
        sanity_distances
    ) = criterion(

        z1,
        z2,
        target
    )


positive_distances = (

    sanity_distances[
        target == 1
    ]
)


negative_distances = (

    sanity_distances[
        target == 0
    ]
)


print("=" * 76)
print("SIAMESE RESNET — INITIAL SANITY CHECK")
print("=" * 76)

print(
    "x1:",
    x1.shape
)

print(
    "Embedding:",
    z1.shape
)

print()

print(
    "Loss:",
    f"{sanity_loss.item():.4f}"
)

print(
    "Mean d+:",
    f"{positive_distances.mean().item():.4f}"
)

print(
    "Mean d-:",
    f"{negative_distances.mean().item():.4f}"
)

print(
    "Initial gap:",
    f"{(
        negative_distances.mean()
        -
        positive_distances.mean()
    ).item():.4f}"
)

print()

print(
    "z1 norm:",
    f"{z1.norm(dim=1).mean().item():.4f}"
)

print(
    "z2 norm:",
    f"{z2.norm(dim=1).mean().item():.4f}"
)

print(
    "Finite loss:",
    torch.isfinite(
        sanity_loss
    ).item()
)


assert (
    z1.shape[1]
    ==
    EMBEDDING_DIM
)


assert torch.isfinite(
    sanity_loss
)


print()
print(
    "Siamese ResNet sanity check: OK"
)

SIAMESE RESNET — INITIAL SANITY CHECK
x1: torch.Size([128, 1, 64, 64])
Embedding: torch.Size([128, 128])

Loss: 0.6367
Mean d+: 0.1257
Mean d-: 0.1385
Initial gap: 0.0128

z1 norm: 1.0000
z2 norm: 1.0000
Finite loss: True

Siamese ResNet sanity check: OK


In [14]:
# ============================================================
# CELL 15 — PAIRWISE EVALUATION
# ============================================================

def evaluate_pairwise(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_targets = []
    all_distances = []


    with torch.no_grad():

        for batch in loader:

            x1 = (
                batch["x1"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            x2 = (
                batch["x2"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            target = (
                batch["target"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=DEVICE.type,

                dtype=(
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16
                ),

                enabled=AMP_ENABLED
            ):

                z1, z2 = model(
                    x1,
                    x2
                )


            loss, distances = criterion(

                z1,
                z2,
                target
            )


            batch_size = (
                target.shape[0]
            )


            total_loss += (
                loss.item()
                *
                batch_size
            )

            total_samples += (
                batch_size
            )


            all_targets.append(
                target
                .cpu()
                .numpy()
            )

            all_distances.append(
                distances
                .cpu()
                .numpy()
            )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    positive_distances = (
        distances[
            targets == 1
        ]
    )

    negative_distances = (
        distances[
            targets == 0
        ]
    )


    d_pos = float(
        positive_distances.mean()
    )

    d_neg = float(
        negative_distances.mean()
    )


    gap = (
        d_neg
        -
        d_pos
    )


    pooled_variance = (
        0.5
        *
        (
            positive_distances.var()
            +
            negative_distances.var()
        )
    )


    d_prime = float(

        gap

        /

        np.sqrt(
            pooled_variance
            +
            1e-12
        )
    )


    auc = float(

        roc_auc_score(
            targets,
            -distances
        )
    )


    return {

        "loss":
            total_loss
            /
            total_samples,

        "auc":
            auc,

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "gap":
            gap,

        "d_prime":
            d_prime
    }

In [23]:
# ============================================================
# CELL 16 — OPTIMIZER + AMP
# ============================================================

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4


try:

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY,

        fused=(
            DEVICE.type
            ==
            "cuda"
        )
    )


    fused_adamw = (
        DEVICE.type
        ==
        "cuda"
    )


except (
    RuntimeError,
    TypeError
):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


    fused_adamw = False


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=AMP_ENABLED
)


print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "AMP:",
    AMP_ENABLED
)

print(
    "Fused AdamW:",
    fused_adamw
)

Learning rate: 0.0003
Weight decay: 0.0001
AMP: True
Fused AdamW: True


In [16]:
# ============================================================
# CELL 17 — TRAIN ONE SIAMESE RESNET EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    dataset,
    criterion,
    optimizer,
    scaler,
    epoch
):

    model.train()


    # Nuove coppie ad ogni epoca
    dataset.set_epoch(
        epoch
    )


    total_loss = 0.0
    total_samples = 0

    all_targets = []
    all_distances = []


    start_time = (
        time.perf_counter()
    )


    for batch in loader:

        x1 = (
            batch["x1"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        x2 = (
            batch["x2"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        target = (
            batch["target"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(

            device_type=DEVICE.type,

            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),

            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        loss, distances = criterion(

            z1,
            z2,
            target
        )


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            optimizer.step()


        batch_size = (
            target.shape[0]
        )


        total_loss += (
            loss.detach().item()
            *
            batch_size
        )

        total_samples += (
            batch_size
        )


        all_targets.append(

            target
            .detach()
            .cpu()
            .numpy()
        )

        all_distances.append(

            distances
            .detach()
            .cpu()
            .numpy()
        )


    elapsed = (
        time.perf_counter()
        -
        start_time
    )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    train_auc = float(

        roc_auc_score(
            targets,
            -distances
        )
    )


    positive_distances = (
        distances[
            targets == 1
        ]
    )

    negative_distances = (
        distances[
            targets == 0
        ]
    )


    return {

        "loss":
            total_loss
            /
            total_samples,

        "auc":
            train_auc,

        "d_pos":
            float(
                positive_distances.mean()
            ),

        "d_neg":
            float(
                negative_distances.mean()
            ),

        "gap":
            float(
                negative_distances.mean()
                -
                positive_distances.mean()
            ),

        "seconds":
            elapsed
    }

In [18]:
# ============================================================
# CELL 18 — SAVE INITIAL STATE
# ============================================================

initial_resnet_state = copy.deepcopy(
    model.state_dict()
)


print(
    "Initial ResNet state salvato."
)

Initial ResNet state salvato.


In [24]:
# ============================================================
# COMPACT RESNET SPEED BENCHMARK
# ============================================================

benchmark_model_state = copy.deepcopy(
    model.state_dict()
)


benchmark_batches = 20


if DEVICE.type == "cuda":

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


model.train()


start = time.perf_counter()


for batch_idx, batch in enumerate(
    train_pair_loader
):

    if batch_idx >= benchmark_batches:
        break


    x1 = (
        batch["x1"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    x2 = (
        batch["x2"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    target = (
        batch["target"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    with torch.autocast(

        device_type=DEVICE.type,

        dtype=(
            torch.float16
            if DEVICE.type == "cuda"
            else torch.bfloat16
        ),

        enabled=AMP_ENABLED

    ):

        z1, z2 = model(
            x1,
            x2
        )


    loss, _ = criterion(
        z1,
        z2,
        target
    )


    if AMP_ENABLED:

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

    else:

        loss.backward()

        optimizer.step()


if DEVICE.type == "cuda":

    torch.cuda.synchronize()


elapsed = (
    time.perf_counter()
    -
    start
)


seconds_per_batch = (
    elapsed
    /
    benchmark_batches
)


estimated_epoch_seconds = (

    seconds_per_batch

    *

    len(
        train_pair_loader
    )
)


print("=" * 72)
print("COMPACT RESNET SPEED BENCHMARK")
print("=" * 72)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Batches / epoch:",
    len(train_pair_loader)
)

print(
    "Seconds / batch:",
    f"{seconds_per_batch:.3f}"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch_seconds:.1f} s"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch_seconds / 60:.2f} min"
)


if DEVICE.type == "cuda":

    peak_memory_gb = (

        torch.cuda
        .max_memory_allocated()

        /

        1024**3
    )


    print(
        "Peak GPU memory:",
        f"{peak_memory_gb:.2f} GB"
    )


# ============================================================
# RESTORE MODEL
# ============================================================

model.load_state_dict(
    benchmark_model_state
)


# ============================================================
# RESET OPTIMIZER
# ============================================================

try:

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY,

        fused=(
            DEVICE.type == "cuda"
        )
    )

except (RuntimeError, TypeError):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


print()
print(
    "Model + optimizer ripristinati dopo benchmark."
)

COMPACT RESNET SPEED BENCHMARK
Batch size: 128
Batches / epoch: 391
Seconds / batch: 0.151
Estimated epoch: 58.9 s
Estimated epoch: 0.98 min
Peak GPU memory: 1.74 GB

Model + optimizer ripristinati dopo benchmark.


In [19]:
# ============================================================
# SPEED + GPU MEMORY BENCHMARK
# ============================================================

model.train()

torch.cuda.empty_cache()

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()


benchmark_batches = 20

start = time.perf_counter()


for batch_idx, batch in enumerate(train_pair_loader):

    if batch_idx >= benchmark_batches:
        break

    x1 = batch["x1"].to(
        DEVICE,
        non_blocking=True
    )

    x2 = batch["x2"].to(
        DEVICE,
        non_blocking=True
    )

    target = batch["target"].to(
        DEVICE,
        non_blocking=True
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type=DEVICE.type,
        dtype=(
            torch.float16
            if DEVICE.type == "cuda"
            else torch.bfloat16
        ),
        enabled=AMP_ENABLED
    ):

        z1, z2 = model(
            x1,
            x2
        )

    loss, _ = criterion(
        z1,
        z2,
        target
    )

    if AMP_ENABLED:

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

    else:

        loss.backward()
        optimizer.step()


if DEVICE.type == "cuda":
    torch.cuda.synchronize()


elapsed = (
    time.perf_counter()
    -
    start
)


seconds_per_batch = (
    elapsed
    /
    benchmark_batches
)


estimated_epoch_seconds = (
    seconds_per_batch
    *
    len(train_pair_loader)
)


print("=" * 70)
print("RESNET SPEED BENCHMARK")
print("=" * 70)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Batch per epoch:",
    len(train_pair_loader)
)

print(
    "Seconds / batch:",
    f"{seconds_per_batch:.3f}"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch_seconds:.1f} s"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch_seconds / 60:.2f} min"
)


if DEVICE.type == "cuda":

    peak_memory_gb = (
        torch.cuda.max_memory_allocated()
        /
        1024**3
    )

    print(
        "Peak GPU memory:",
        f"{peak_memory_gb:.2f} GB"
    )

RESNET SPEED BENCHMARK
Batch size: 128
Batch per epoch: 391
Seconds / batch: 0.611
Estimated epoch: 239.0 s
Estimated epoch: 3.98 min
Peak GPU memory: 3.00 GB


In [25]:
# ============================================================
# CELL 19 — COMPACT SIAMESE RESNET SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


print("=" * 100)
print("COMPACT SIAMESE RESNET — EUCLIDEAN CONTRASTIVE — SMOKE TEST")
print("=" * 100)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = train_one_epoch(

        model=model,

        loader=train_pair_loader,

        dataset=train_pair_dataset,

        criterion=criterion,

        optimizer=optimizer,

        scaler=scaler,

        epoch=epoch
    )


    val_metrics = evaluate_pairwise(

        model=model,

        loader=val_pair_loader,

        criterion=criterion
    )


    print(

        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

COMPACT SIAMESE RESNET — EUCLIDEAN CONTRASTIVE — SMOKE TEST
Epoch 01/3 | train loss 0.3843 | train AUC 0.6251 | val loss 0.3779 | val AUC 0.6292 | d+ 0.5430 | d- 0.6429 | gap 0.0999 | d' 0.4647 | 52.0s
Epoch 02/3 | train loss 0.3701 | train AUC 0.6424 | val loss 0.3823 | val AUC 0.6408 | d+ 0.6298 | d- 0.7555 | gap 0.1257 | d' 0.5058 | 160.5s
Epoch 03/3 | train loss 0.3676 | train AUC 0.6474 | val loss 0.3709 | val AUC 0.6425 | d+ 0.5985 | d- 0.7048 | gap 0.1063 | d' 0.5116 | 88.4s


In [26]:
# ============================================================
# CELL 20 — CLEAN RESET BEFORE FULL TRAINING
# ============================================================

model.load_state_dict(
    benchmark_model_state,
    strict=True
)


# Nuovo optimizer pulito
try:

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY,

        fused=(DEVICE.type == "cuda")
    )

except (RuntimeError, TypeError):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


print(
    "Compact ResNet ripristinata allo stato pre-smoke."
)

print(
    "Optimizer e scaler reinizializzati."
)

Compact ResNet ripristinata allo stato pre-smoke.
Optimizer e scaler reinizializzati.


In [27]:
# ============================================================
# CELL 21 — FULL TRAINING CONFIG
# ============================================================

MAX_EPOCHS = 40

EARLY_STOPPING_PATIENCE = 10

MIN_DELTA = 1e-4


BEST_CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "compact_resnet_euclidean_margin_1p25_best.pt"
)


HISTORY_PATH = (
    ARTIFACTS_DIR
    / "compact_resnet_euclidean_history.tsv"
)


print(
    "Max epochs:",
    MAX_EPOCHS
)

print(
    "Patience:",
    EARLY_STOPPING_PATIENCE
)

print(
    "Checkpoint metric: pairwise validation ROC-AUC"
)

print(
    "Checkpoint:",
    BEST_CHECKPOINT_PATH
)

Max epochs: 40
Patience: 10
Checkpoint metric: pairwise validation ROC-AUC
Checkpoint: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\siamese_resnet_tumor_healthy\compact_resnet_euclidean_margin_1p25_best.pt


In [28]:
# ============================================================
# CELL 22 — FULL COMPACT RESNET TRAINING
# ============================================================

best_val_auc = -np.inf

best_epoch = 0

epochs_without_improvement = 0

history = []


print("=" * 104)
print("COMPACT SIAMESE RESNET — EUCLIDEAN CONTRASTIVE — FULL TRAINING")
print("=" * 104)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # ========================================================
    # TRAIN
    # ========================================================

    train_metrics = train_one_epoch(

        model=model,

        loader=train_pair_loader,

        dataset=train_pair_dataset,

        criterion=criterion,

        optimizer=optimizer,

        scaler=scaler,

        epoch=epoch
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_metrics = evaluate_pairwise(

        model=model,

        loader=val_pair_loader,

        criterion=criterion
    )


    current_auc = float(
        val_metrics["auc"]
    )


    improved = (
        current_auc
        >
        best_val_auc + MIN_DELTA
    )


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    if improved:

        best_val_auc = (
            current_auc
        )

        best_epoch = (
            epoch
        )

        epochs_without_improvement = 0


        torch.save(
            {
                "epoch":
                    epoch,

                "best_epoch":
                    epoch,

                "best_val_auc":
                    current_auc,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "embedding_dim":
                    EMBEDDING_DIM,

                "margin":
                    EUCLIDEAN_MARGIN,

                "learning_rate":
                    LEARNING_RATE,

                "weight_decay":
                    WEIGHT_DECAY,

                "batch_size":
                    BATCH_SIZE,

                "k":
                    K,

                "n_classes":
                    N_CLASSES,

                "architecture":
                    "compact_residual_cnn",

                "task":
                    "tumor_healthy",

                "val_loss":
                    val_metrics["loss"],

                "d_pos":
                    val_metrics["d_pos"],

                "d_neg":
                    val_metrics["d_neg"],

                "gap":
                    val_metrics["gap"],

                "d_prime":
                    val_metrics["d_prime"]
            },

            BEST_CHECKPOINT_PATH
        )


    else:

        epochs_without_improvement += 1


    # ========================================================
    # HISTORY
    # ========================================================

    history.append(
        {
            "epoch":
                epoch,

            "train_loss":
                train_metrics["loss"],

            "train_auc":
                train_metrics["auc"],

            "val_loss":
                val_metrics["loss"],

            "val_auc":
                val_metrics["auc"],

            "d_pos":
                val_metrics["d_pos"],

            "d_neg":
                val_metrics["d_neg"],

            "gap":
                val_metrics["gap"],

            "d_prime":
                val_metrics["d_prime"],

            "seconds":
                train_metrics["seconds"]
        }
    )


    marker = (
        " *BEST*"
        if improved
        else ""
    )


    print(

        f"Epoch {epoch:02d}/{MAX_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"

        f"{marker}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print()

        print(
            f"Early stopping at epoch {epoch}."
        )

        break


# ============================================================
# SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(

    HISTORY_PATH,

    sep="\t",

    index=False
)


print()
print("=" * 104)
print("TRAINING COMPLETATO")
print("=" * 104)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best Val ROC-AUC:",
    f"{best_val_auc:.6f}"
)

print(
    "Checkpoint:",
    BEST_CHECKPOINT_PATH
)

COMPACT SIAMESE RESNET — EUCLIDEAN CONTRASTIVE — FULL TRAINING
Epoch 01/40 | train loss 0.3843 | train AUC 0.6251 | val loss 0.3779 | val AUC 0.6292 | d+ 0.5430 | d- 0.6429 | gap 0.0999 | d' 0.4647 | 42.2s *BEST*
Epoch 02/40 | train loss 0.3701 | train AUC 0.6424 | val loss 0.3823 | val AUC 0.6408 | d+ 0.6298 | d- 0.7555 | gap 0.1257 | d' 0.5058 | 42.8s *BEST*
Epoch 03/40 | train loss 0.3676 | train AUC 0.6474 | val loss 0.3709 | val AUC 0.6425 | d+ 0.5985 | d- 0.7048 | gap 0.1063 | d' 0.5116 | 97.8s *BEST*
Epoch 04/40 | train loss 0.3662 | train AUC 0.6499 | val loss 0.3759 | val AUC 0.6357 | d+ 0.5113 | d- 0.6062 | gap 0.0948 | d' 0.4868 | 111.0s
Epoch 05/40 | train loss 0.3618 | train AUC 0.6588 | val loss 0.3777 | val AUC 0.6238 | d+ 0.5841 | d- 0.6759 | gap 0.0918 | d' 0.4463 | 207.9s
Epoch 06/40 | train loss 0.3612 | train AUC 0.6605 | val loss 0.3694 | val AUC 0.6403 | d+ 0.5481 | d- 0.6416 | gap 0.0935 | d' 0.5062 | 51.8s
Epoch 07/40 | train loss 0.3597 | train AUC 0.6644 | val

In [29]:
# ============================================================
# CELL 23 — BEST COMPACT RESNET CHECKPOINT
# ============================================================

best_checkpoint = torch.load(

    BEST_CHECKPOINT_PATH,

    map_location=DEVICE
)


model.load_state_dict(

    best_checkpoint[
        "model_state_dict"
    ],

    strict=True
)


best_pairwise_metrics = evaluate_pairwise(

    model=model,

    loader=val_pair_loader,

    criterion=criterion
)


print("=" * 76)
print("BEST COMPACT SIAMESE RESNET")
print("=" * 76)

print(
    "Best epoch:",
    best_checkpoint["best_epoch"]
)

print(
    "Val ROC-AUC:",
    f"{best_pairwise_metrics['auc']:.6f}"
)

print(
    "Val loss:",
    f"{best_pairwise_metrics['loss']:.6f}"
)

print(
    "d+:",
    f"{best_pairwise_metrics['d_pos']:.6f}"
)

print(
    "d-:",
    f"{best_pairwise_metrics['d_neg']:.6f}"
)

print(
    "Gap:",
    f"{best_pairwise_metrics['gap']:.6f}"
)

print(
    "d-prime:",
    f"{best_pairwise_metrics['d_prime']:.6f}"
)

BEST COMPACT SIAMESE RESNET
Best epoch: 19
Val ROC-AUC: 0.656950
Val loss: 0.364024
d+: 0.572312
d-: 0.692110
Gap: 0.119798
d-prime: 0.567054


In [30]:
# ============================================================
# CELL 24 — SINGLE FCGR DATASET FOR EMBEDDINGS
# ============================================================

class SingleFCGRDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )

        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


    def __len__(self):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        row = int(
            self.rows[index]
        )

        x = np.array(

            self.fcgr_memmap[row],

            dtype=np.float32,

            copy=True
        )


        return {

            "x":
                torch.from_numpy(
                    x
                ).unsqueeze(0),

            "class_id":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }

In [31]:
# ============================================================
# CELL 25 — EXTRACT EMBEDDINGS
# ============================================================

reference_dataset = SingleFCGRDataset(

    metadata=
        full_train_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


validation_dataset = SingleFCGRDataset(

    metadata=
        val_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


reference_loader = DataLoader(

    reference_dataset,

    batch_size=256,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


validation_loader = DataLoader(

    validation_dataset,

    batch_size=256,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


def extract_embeddings(
    model,
    loader
):

    model.eval()

    embeddings = []
    labels = []


    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=DEVICE.type,

                dtype=(
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16
                ),

                enabled=AMP_ENABLED
            ):

                z = model.encoder(
                    x
                )


            embeddings.append(

                z.float()
                .cpu()
                .numpy()
            )


            labels.append(

                batch["class_id"]
                .numpy()
            )


    return (
        np.concatenate(
            embeddings,
            axis=0
        ),

        np.concatenate(
            labels,
            axis=0
        )
    )


reference_embeddings, reference_labels = (
    extract_embeddings(
        model,
        reference_loader
    )
)


val_embeddings, val_labels = (
    extract_embeddings(
        model,
        validation_loader
    )
)


print(
    "Reference embeddings:",
    reference_embeddings.shape
)

print(
    "Validation embeddings:",
    val_embeddings.shape
)

print(
    "Embedding norm:",
    np.linalg.norm(
        reference_embeddings,
        axis=1
    ).mean()
)

Reference embeddings: (96167, 128)
Validation embeddings: (9753, 128)
Embedding norm: 1.0


In [32]:
# ============================================================
# CELL 26 — EUCLIDEAN PROTOTYPE CLASSIFICATION
# ============================================================

prototypes = []


for class_id in range(
    N_CLASSES
):

    class_embeddings = (
        reference_embeddings[
            reference_labels
            ==
            class_id
        ]
    )


    prototype = (
        class_embeddings
        .mean(axis=0)
    )


    # Normalizziamo il centro della classe
    prototype = (

        prototype

        /

        (
            np.linalg.norm(
                prototype
            )
            +
            1e-12
        )
    )


    prototypes.append(
        prototype
    )


prototypes = np.stack(
    prototypes,
    axis=0
)


# ============================================================
# DISTANCE:
#
# val samples × 12 prototypes
# ============================================================

distances = np.linalg.norm(

    val_embeddings[:, None, :]

    -

    prototypes[None, :, :],

    axis=2
)


val_predictions = (
    distances.argmin(
        axis=1
    )
)


accuracy = accuracy_score(

    val_labels,
    val_predictions
)


macro_f1 = f1_score(

    val_labels,
    val_predictions,

    average="macro",

    zero_division=0
)


balanced_acc = balanced_accuracy_score(

    val_labels,
    val_predictions
)


print("=" * 76)
print("COMPACT RESNET — 12-WAY PROTOTYPE CLASSIFICATION")
print("=" * 76)

print(
    "Accuracy:",
    f"{accuracy:.6f}"
)

print(
    "Macro-F1:",
    f"{macro_f1:.6f}"
)

print(
    "Balanced Accuracy:",
    f"{balanced_acc:.6f}"
)

COMPACT RESNET — 12-WAY PROTOTYPE CLASSIFICATION
Accuracy: 0.269968
Macro-F1: 0.235231
Balanced Accuracy: 0.290067
